In [11]:
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path("..")
INTERIM_PATH = PROJECT_ROOT / "data" / "interim" / "order_level_clean.csv"

df = pd.read_csv(
    INTERIM_PATH,
    parse_dates=["order date (DateOrders)"]
)

print(df.shape)
print(df.head())

(65752, 15)
   Order Id     Type Customer Segment Customer State Order Country  \
0         1     CASH         Consumer             NC        México   
1         2  PAYMENT         Consumer             IL      Colombia   
2         4     CASH      Home Office             TX      Colombia   
3         5    DEBIT         Consumer             PR      Colombia   
4         7    DEBIT         Consumer             FL        Brasil   

      Order Region   Shipping Mode  Customer Id order date (DateOrders)  \
0  Central America  Standard Class        11599     2015-01-01 00:00:00   
1    South America  Standard Class          256     2015-01-01 00:21:00   
2    South America  Standard Class         8827     2015-01-01 01:03:00   
3    South America  Standard Class        11318     2015-01-01 01:24:00   
4    South America    Second Class         4530     2015-01-01 02:06:00   

   Late_delivery_risk  total_quantity  total_discount  num_unique_products  \
0                   0               1 

In [12]:
print("Shape:", df.shape)
print("Unique orders:", df["Order Id"].nunique())
print("Date range:")
print(
    df["order date (DateOrders)"].min(),
    "→",
    df["order date (DateOrders)"].max()
)

Shape: (65752, 15)
Unique orders: 65752
Date range:
2015-01-01 00:00:00 → 2018-01-31 23:38:00


In [13]:
df = df.sort_values("order date (DateOrders)").reset_index(drop=True)

In [14]:
n = len(df)

train_end = int(n * 0.70)
val_end = int(n * 0.85)

train_df = df.iloc[:train_end].copy()
val_df = df.iloc[train_end:val_end].copy()
test_df = df.iloc[val_end:].copy()

In [15]:
for name, split_df in [
    ("Train", train_df),
    ("Validation", val_df),
    ("Test", test_df),
]:
    print(f"\n{name}")
    print("Rows:", len(split_df))
    print(
        "Date range:",
        split_df["order date (DateOrders)"].min(),
        "→",
        split_df["order date (DateOrders)"].max()
    )
    print(
        "Late rate:",
        round(split_df["Late_delivery_risk"].mean(), 4)
    )


Train
Rows: 46026
Date range: 2015-01-01 00:00:00 → 2017-03-16 19:22:00
Late rate: 0.5485

Validation
Rows: 9863
Date range: 2017-03-16 19:43:00 → 2017-09-05 12:38:00
Late rate: 0.545

Test
Rows: 9863
Date range: 2017-09-05 12:59:00 → 2018-01-31 23:38:00
Late rate: 0.5505


In [16]:
train_customers = set(train_df["Customer Id"])
val_customers = set(val_df["Customer Id"])
test_customers = set(test_df["Customer Id"])

print("Train ↔ Validation overlap:", len(train_customers & val_customers))
print("Train ↔ Test overlap:", len(train_customers & test_customers))
print("Validation ↔ Test overlap:", len(val_customers & test_customers))

Train ↔ Validation overlap: 6670
Train ↔ Test overlap: 1424
Validation ↔ Test overlap: 808


In [17]:
new_val_customers = val_customers - train_customers
new_test_customers = test_customers - train_customers

print("New customers in Validation:", len(new_val_customers))
print("New customers in Test:", len(new_test_customers))

New customers in Validation: 167
New customers in Test: 8355


In [18]:
print("Train unique customers:", train_df["Customer Id"].nunique())
print("Validation unique customers:", val_df["Customer Id"].nunique())
print("Test unique customers:", test_df["Customer Id"].nunique())

Train unique customers: 12152
Validation unique customers: 6837
Test unique customers: 9779


In [19]:
val_new_customer_pct = (
    len(new_val_customers) / val_df["Customer Id"].nunique() * 100
)

test_new_customer_pct = (
    len(new_test_customers) / test_df["Customer Id"].nunique() * 100
)

print("New customer % in Validation:", round(val_new_customer_pct, 2))
print("New customer % in Test:", round(test_new_customer_pct, 2))

New customer % in Validation: 2.44
New customer % in Test: 85.44


In [21]:
SPLIT_DIR = PROJECT_ROOT / "data" / "interim" / "splits"
SPLIT_DIR.mkdir(parents=True, exist_ok=True)

train_df.to_csv(SPLIT_DIR / "train.csv", index=False)
val_df.to_csv(SPLIT_DIR / "validation.csv", index=False)
test_df.to_csv(SPLIT_DIR / "test.csv", index=False)